In [3]:
import pandas as pd
import numpy as np
import mysql.connector
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_df(sql):
    conn = get_conn()
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

df = fetch_df("""
    SELECT neo_id, full_name, designation_num, is_hazardous, eccentricity,
           semi_major_axis_au, inclination_deg, orbital_period_days,
           impact_probability, palermo_scale_max, torino_scale
    FROM dim_neo
""")
print(df.shape)
df.head()

(2760, 11)


C:\Users\HP\AppData\Local\Temp\ipykernel_10200\1361894321.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,neo_id,full_name,designation_num,is_hazardous,eccentricity,semi_major_axis_au,inclination_deg,orbital_period_days,impact_probability,palermo_scale_max,torino_scale
0,20000433,433 Eros (A898 PA),433,NaN,0.2229,1.458,10.83,643.0,None,None,None
1,20000719,719 Albert (A911 TB),719,NaN,0.5466,2.637,11.57,1560.0,None,None,None
2,20000887,887 Alinda (A918 AA),887,NaN,0.5711,2.474,9.40,1420.0,None,None,None
3,20001036,1036 Ganymed (A924 UB),1036,NaN,0.5335,2.664,26.69,1590.0,None,None,None
4,20001221,1221 Amor (1932 EA1),1221,NaN,0.4347,1.920,11.87,972.0,None,None,None


In [4]:
features = ["eccentricity", "semi_major_axis_au", "inclination_deg", "orbital_period_days"]
model_df = df[features].fillna(df[features].median())

scaler = StandardScaler()
X = scaler.fit_transform(model_df)

iso = IsolationForest(contamination=0.05, random_state=42)
df["anomaly_flag"] = iso.fit_predict(X)          # -1 = anomaly, 1 = normal
df["anomaly_score"] = -iso.score_samples(X)       # higher = more anomalous

df["is_anomaly"] = (df["anomaly_flag"] == -1).astype(int)
print(df["is_anomaly"].value_counts())
df.sort_values("anomaly_score", ascending=False)[["full_name","anomaly_score"] + features].head(10)

is_anomaly
0    2622
1     138
Name: count, dtype: int64


,full_name,anomaly_score,eccentricity,semi_major_axis_au,inclination_deg,orbital_period_days
878,196256 (2003 EH1),0.720111,0.6187,3.123,70.87,2020.0
48,3552 Don Quixote (1983 SA),0.700193,0.7074,4.271,31.05,3220.0
342,85490 (1997 SE5),0.684040,0.6599,3.765,2.58,2670.0
1698,410778 (2009 FG19),0.681693,0.7189,2.910,54.50,1810.0
1017,248590 (2006 CS),0.675400,0.6970,2.916,52.31,1820.0
1356,343158 Marsyas (2009 HC82),0.672931,0.8068,2.527,154.35,1470.0
1922,437994 (2003 UL12),0.670718,0.7034,3.279,19.82,2170.0
1723,414287 (2008 OB9),0.670106,0.7597,3.201,13.50,2090.0
88,5370 Taranis (1986 RA),0.666987,0.6372,3.318,19.18,2210.0
1415,355256 Margarethekahn (2007 KN4),0.664627,0.6298,3.348,12.52,2240.0


In [5]:
df["hazard_component"] = df["is_hazardous"].fillna(0).astype(float)
df["impact_component"] = (df["impact_probability"].fillna(0) / (df["impact_probability"].max() or 1))
df["torino_component"] = (df["torino_scale"].fillna(0) / 10)
df["anomaly_component"] = (df["anomaly_score"] - df["anomaly_score"].min()) / \
                           (df["anomaly_score"].max() - df["anomaly_score"].min())

df["ml_risk_score"] = (
    df["hazard_component"] * 0.35 +
    df["impact_component"] * 0.35 +
    df["torino_component"] * 0.20 +
    df["anomaly_component"] * 0.10
) * 100

df.sort_values("ml_risk_score", ascending=False)[["full_name","ml_risk_score"]].head(10)

C:\Users\HP\AppData\Local\Temp\ipykernel_10200\964551488.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["impact_component"] = (df["impact_probability"].fillna(0) / (df["impact_probability"].max() or 1))
C:\Users\HP\AppData\Local\Temp\ipykernel_10200\964551488.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["torino_component"] = (df["torino_scale"].fillna(0) / 10)


,full_name,ml_risk_score
0,433 Eros (A898 PA),NaN
1,719 Albert (A911 TB),NaN
2,887 Alinda (A918 AA),NaN
3,1036 Ganymed (A924 UB),NaN
4,1221 Amor (1932 EA1),NaN
5,1566 Icarus (1949 MA),NaN
6,1580 Betulia (1950 KA),NaN
7,1620 Geographos (1951 RA),NaN
8,1627 Ivar (1929 SH),NaN
9,1685 Toro (1948 OA),NaN


In [8]:
def write_ai_results(df):
    conn = get_conn()
    cur = conn.cursor()

    for col, coltype in [("is_anomaly", "INT"), ("anomaly_score", "FLOAT"), ("ml_risk_score", "FLOAT")]:
        try:
            cur.execute(f"ALTER TABLE dim_neo ADD COLUMN {col} {coltype}")
            conn.commit()
        except mysql.connector.errors.ProgrammingError as e:
            if "Duplicate column name" in str(e):
                pass
            else:
                raise

    df_clean = df.replace({np.nan: None})

    for _, row in df_clean.iterrows():
        if row["neo_id"] is None:
            continue  # skip rows with no valid ID
        cur.execute("""
            UPDATE dim_neo
            SET is_anomaly = %s, anomaly_score = %s, ml_risk_score = %s
            WHERE neo_id = %s
        """, (
            None if row["is_anomaly"] is None else int(row["is_anomaly"]),
            None if row["anomaly_score"] is None else float(row["anomaly_score"]),
            None if row["ml_risk_score"] is None else float(row["ml_risk_score"]),
            row["neo_id"]
        ))
    conn.commit()
    cur.close(); conn.close()

write_ai_results(df)
print("AI results written to MySQL.")

AI results written to MySQL.


In [11]:
check = fetch_df("""
    SELECT full_name, is_anomaly, anomaly_score, ml_risk_score
    FROM dim_neo ORDER BY ml_risk_score DESC LIMIT 10
""")
print(check)

                   full_name  is_anomaly  anomaly_score ml_risk_score
0         433 Eros (A898 PA)           0       0.439607          None
1       719 Albert (A911 TB)           0       0.504790          None
2       887 Alinda (A918 AA)           0       0.467102          None
3     1036 Ganymed (A924 UB)           0       0.529947          None
4       1221 Amor (1932 EA1)           0       0.429426          None
5      1566 Icarus (1949 MA)           0       0.549833          None
6     1580 Betulia (1950 KA)           1       0.581263          None
7  1620 Geographos (1951 RA)           0       0.412140          None
8        1627 Ivar (1929 SH)           0       0.430932          None
9        1685 Toro (1948 OA)           0       0.416882          None


C:\Users\HP\AppData\Local\Temp\ipykernel_10200\1361894321.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


In [12]:
max_ip = df["impact_probability"].max()
max_ip = 1 if pd.isna(max_ip) or max_ip == 0 else max_ip
df["impact_component"] = df["impact_probability"].fillna(0) / max_ip

df["torino_component"] = (df["torino_scale"].fillna(0) / 10)
df["anomaly_component"] = (df["anomaly_score"] - df["anomaly_score"].min()) / \
                           (df["anomaly_score"].max() - df["anomaly_score"].min())

df["ml_risk_score"] = (
    df["hazard_component"] * 0.35 +
    df["impact_component"] * 0.35 +
    df["torino_component"] * 0.20 +
    df["anomaly_component"] * 0.10
) * 100

df[["full_name","ml_risk_score"]].sort_values("ml_risk_score", ascending=False).head(10)

C:\Users\HP\AppData\Local\Temp\ipykernel_10200\309579369.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["impact_component"] = df["impact_probability"].fillna(0) / max_ip
C:\Users\HP\AppData\Local\Temp\ipykernel_10200\309579369.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["torino_component"] = (df["torino_scale"].fillna(0) / 10)


,full_name,ml_risk_score
2243,(2011 YE6),41.132541
2092,(2001 EC),40.899327
1916,437844 (1999 MN),40.661338
2314,(2014 JS54),40.632070
1094,276033 (2002 AJ129),40.213597
1523,374038 (2004 HW),40.089335
2012,500080 (2011 WV134),39.948923
2378,(2015 TD323),39.851154
2320,(2014 MR26),39.698011
1005,242708 (2005 UK1),39.647666


In [13]:
write_ai_results(df)
print("AI results written to MySQL.")

AI results written to MySQL.


In [14]:
check = fetch_df("""
    SELECT full_name, is_anomaly, anomaly_score, ml_risk_score
    FROM dim_neo ORDER BY ml_risk_score DESC LIMIT 10
""")
print(check)

             full_name  is_anomaly  anomaly_score  ml_risk_score
0           (2011 YE6)           1       0.599395        41.1325
1            (2001 EC)           1       0.592116        40.8993
2     437844 (1999 MN)           1       0.584687        40.6613
3          (2014 JS54)           1       0.583774        40.6321
4  276033 (2002 AJ129)           0       0.570712        40.2136
5     374038 (2004 HW)           0       0.566833        40.0893
6  500080 (2011 WV134)           0       0.562451        39.9489
7         (2015 TD323)           0       0.559399        39.8512
8          (2014 MR26)           0       0.554619        39.6980
9    242708 (2005 UK1)           0       0.553047        39.6477


C:\Users\HP\AppData\Local\Temp\ipykernel_10200\1361894321.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)
